# Stage 1 — Data Ingestion

**Goal of this stage:** land the raw movie data locally, faithfully, with *no* cleaning or filtering. Cleaning happens in Data Validation (Stage 2) and filtering/feature work in Data Transformation (Stage 3).

**Source decision:** the **`movies`** table in Supabase — the db1-rich dataset (~85k rows, with `overview`, `genres`, `movie_cast`, `director`, `writers`, ratings, etc.). We deliberately do **not** use `movies2` (the feature-light db2 dataset, ~9k rows).

**Project decision being served:** *"What should I watch next?"* — a content-based recommender. Rich item attributes (especially `overview` + `genres`) are what make content-based recommendations good, which is why this table is the spine.

This notebook prototypes the ingestion logic; the same logic is then ported into `src/Recommender_system/components/data_ingestion.py`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

# Run from the project root so relative paths line up with the pipeline.
if Path.cwd().name == "research":
    os.chdir("..")
print("cwd:", Path.cwd())

RAW_PATH = "artifacts/data_ingestion/movies_raw.csv"
SOURCE_TABLE = "movies"
BATCH_SIZE = 10_000

## 1. Connecting to Supabase

Credentials live in `.env` (gitignored) and are read as environment variables — never hard-coded. We use the same `psycopg2` connection as `ping_to_supabase.ipynb` (Supabase transaction pooler).

In [ ]:
def get_connection():
    load_dotenv()
    return psycopg2.connect(
        user=os.getenv("user"),
        password=os.getenv("password"),
        host=os.getenv("host"),
        port=os.getenv("port"),
        dbname=os.getenv("dbname"),
        connect_timeout=30,
    )

# Quick health check
conn = get_connection()
cur = conn.cursor()
cur.execute(f'SELECT COUNT(*) FROM public."{SOURCE_TABLE}";')
print("rows in source table:", cur.fetchone()[0])
cur.close(); conn.close()

## 2. Why batched reads

A single `SELECT *` over all ~85k rich rows overruns the Supabase transaction-pooler **statement timeout** (the connection drops mid-query with `SSL connection has been closed unexpectedly`).

Fix: page through the table with `ORDER BY ctid LIMIT … OFFSET …`, opening a fresh connection per batch. `ctid` is a stable, unique physical row id, so pagination can't skip or duplicate rows.

In [ ]:
def fetch_table_in_batches(table: str, batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    frames, columns, offset = [], None, 0
    while True:
        conn = get_connection()
        try:
            cur = conn.cursor()
            cur.execute(
                f'SELECT * FROM public."{table}" ORDER BY ctid LIMIT %s OFFSET %s;',
                (batch_size, offset),
            )
            columns = [d[0] for d in cur.description]
            rows = cur.fetchall()
            cur.close()
        finally:
            conn.close()
        if not rows:
            break
        frames.append(pd.DataFrame(rows, columns=columns))
        offset += len(rows)
        print(f"  fetched {offset:,} rows ...")
        if len(rows) < batch_size:
            break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)

# Pull only if we haven't already saved the raw artifact (the full pull takes a few minutes).
if not os.path.exists(RAW_PATH):
    os.makedirs(os.path.dirname(RAW_PATH), exist_ok=True)
    df_raw = fetch_table_in_batches(SOURCE_TABLE)
    df_raw.to_csv(RAW_PATH, index=False)
    print("saved ->", RAW_PATH)
else:
    print("raw artifact already exists ->", RAW_PATH)

## 3. EDA snapshot of the raw landing

Just enough profiling to know what we're working with. (Decisions about filtering/cleaning belong to later stages — here we only *observe*.)

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print("shape:", df.shape)
print("\ncolumns:")
print(list(df.columns))
df.head()

In [ ]:
# Missing / placeholder values on the columns that drive content similarity.
content_cols = ["title", "overview", "genres", "movie_cast", "director",
                "writers", "tagline", "imdb_rating", "imdb_votes", "poster_path"]
PLACEHOLDERS = {"", "not_found", "NaN", "nan"}
report = []
for c in content_cols:
    nulls = df[c].isna().sum()
    empties = df[c].astype(str).str.strip().isin(PLACEHOLDERS).sum()
    report.append({"column": c,
                   "null": int(nulls),
                   "empty/not_found": int(empties),
                   "% usable": round(100 * (len(df) - nulls - empties) / len(df), 1)})
pd.DataFrame(report)

In [ ]:
# imdb_votes is extremely long-tailed (a -1 sentinel marks 'unknown').
# This previews the filtering lever we'll pull in Stage 3 (Transformation).
votes = pd.to_numeric(df["imdb_votes"], errors="coerce")
print("median votes:", votes.median(), "| mean:", round(votes.mean(), 0), "| max:", votes.max())
for thr in [100, 500, 1000, 5000, 10000, 25000]:
    print(f"  imdb_votes >= {thr:>6}: {(votes >= thr).sum():>6} movies")

In [ ]:
# Duplicate records exist in the source and will need de-duplication later.
print("duplicate imdb_id rows:", int(df["imdb_id"].duplicated().sum()))
print("duplicate title rows  :", int(df["title"].duplicated().sum()))
print("fully-duplicated rows  :", int(df.duplicated().sum()))

## 4. Findings & hand-off

**Ingested:** 85,391 rows × 26 columns from Supabase `movies` → `artifacts/data_ingestion/movies_raw.csv` (~54 MB).

**Healthy for content-based modelling:**
- `overview` — ~0% missing (our primary text signal)
- `title`, `tagline`, `imdb_rating`, `imdb_votes`, `release_date` — clean

**Sparse — to handle in feature engineering (Stage 3):**
- `movie_cast` ~91% missing, `writers` ~64%, `director` ~25%, `genres` ~36% `not_found`
- `poster_path` ~38% missing (affects the deploy UI, not the model)

**Flags for later stages (NOT ingestion's job):**
- **De-duplication needed:** ~26.5k duplicate `imdb_id` rows in the source → handle in Validation/Transformation.
- **Filtering lever:** `imdb_votes` median is ~15 (very long tail; `-1` = unknown). Filtering to a quality subset (e.g. `>= 500` → ~8.4k, `>= 1000` → ~6.2k) both improves recommendation quality and raises feature completeness. Threshold to be chosen in Stage 3.

**Ported to:** `src/Recommender_system/{entity,config,components,pipeline}` — runnable via `python main.py`.